# pSMAD — 01_manifest_qc

**Feeds:** Fig 1e, 1f

**Position in the chain:** run the numbered notebooks in order

Ported unchanged from the original analysis: outputs are as they ran, and no code
cell was edited. Paths appear as `<analysis-root>/...`.


# 01 | Manifest And Metadata QC (Inline)

This notebook extracts metadata from CZI headers and writes a manifest CSV/JSON,
with parsing logic shown directly in notebook cells.


## Cell Guide
1. Set input/output manifest paths.
2. Review inline XML parsing helpers.
3. Parse each CZI into a flat metadata row.
4. Write manifest CSV/JSON.
5. Run dimension/channel QC views.


In [ ]:
import csv
import hashlib
import json
import xml.etree.ElementTree as ET
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 80)

In [ ]:
# -------------------------------
# User configuration
# -------------------------------
ROOT = Path.cwd().resolve()
if not (ROOT / "scripts").exists() and (ROOT.parent / "scripts").exists():
    ROOT = ROOT.parent.resolve()

INPUT_DIR = ROOT / "data/well3-pSMAD-568pSMAD"
OUT_CSV = ROOT / "results/manifests/czi_manifest.csv"
OUT_JSON = ROOT / "results/manifests/czi_manifest.json"

INCLUDE_SHA256 = (
    False  # Slower on large files; enable when strict provenance is needed.
)
WRITE_OUTPUTS = True

print("Project root:", ROOT)
print("Input dir:", INPUT_DIR)
print("Manifest CSV:", OUT_CSV)
print("Manifest JSON:", OUT_JSON)

Project root: <analysis-root>/pSMAD
Input dir: <analysis-root>/pSMAD/data/well3-pSMAD-568pSMAD
Manifest CSV: <analysis-root>/pSMAD/results/manifests/czi_manifest.csv
Manifest JSON: <analysis-root>/pSMAD/results/manifests/czi_manifest.json


## Settings Explained

- `INCLUDE_SHA256`:
  - `False` (faster): skip checksums.
  - `True` (slower): compute per-file SHA-256 for strict provenance tracking.
- `WRITE_OUTPUTS`:
  - `True`: write `czi_manifest.csv` and `czi_manifest.json` for downstream steps.
  - `False`: inspect in-memory table only (debug mode).

Recommended first pass:
- keep `INCLUDE_SHA256=False`
- keep `WRITE_OUTPUTS=True` so downstream notebooks read a stable manifest.


In [ ]:
# -------------------------------
# Metadata parsing helpers
# -------------------------------
XML_START = b"<ImageDocument"
XML_END = b"</ImageDocument>"


def read_embedded_xml(path: Path, max_read_bytes: int = 8_000_000) -> ET.Element:
    # CZI stores XML metadata near file header; parse the ImageDocument block.
    """Extract and parse the embedded ImageDocument XML block from a CZI file header."""
    with path.open("rb") as f:
        blob = f.read(max_read_bytes)

    start = blob.find(XML_START)
    end = blob.find(XML_END)
    if start < 0 or end < 0:
        raise ValueError(f"No CZI ImageDocument XML found near header: {path}")

    xml_bytes = blob[start : end + len(XML_END)]
    return ET.fromstring(xml_bytes)


def xtext(root: ET.Element, xpath: str) -> str | None:
    """Return text content for a metadata XPath, or None when missing."""
    node = root.find(xpath)
    return None if node is None else node.text


def channel_fields(root: ET.Element) -> dict[str, str]:
    """Collect per-channel metadata fields and flatten them into semicolon-separated strings."""
    ids, illum, contrast, ex_nm, em_nm, exposure = [], [], [], [], [], []
    for ch in root.findall("Metadata/Information/Image/Dimensions/Channels/Channel"):
        ids.append(ch.attrib.get("Id", ""))
        illum.append(ch.findtext("IlluminationType") or "")
        contrast.append(ch.findtext("ContrastMethod") or "")
        ex_nm.append(ch.findtext("ExcitationWavelength") or "")
        em_nm.append(ch.findtext("EmissionWavelength") or "")
        exposure.append(ch.findtext("ExposureTime") or "")

    def cat(items):
        return ";".join([x for x in items if x])

    return {
        "channel_ids": cat(ids),
        "channel_illumination": cat(illum),
        "channel_contrast": cat(contrast),
        "channel_excitation_nm": cat(ex_nm),
        "channel_emission_nm": cat(em_nm),
        "channel_exposure_us": cat(exposure),
    }


def scale_axis_um(root: ET.Element, axis_id: str) -> float | None:
    # Zeiss XML scaling is usually stored in meters; convert to um.
    """Read Zeiss scaling metadata for one axis and convert meters to microns."""
    for dist in root.findall("Metadata/Scaling/Items/Distance"):
        if dist.attrib.get("Id") != axis_id:
            continue
        raw = dist.findtext("Value")
        if raw is None:
            return None
        try:
            return float(raw) * 1e6
        except ValueError:
            return None
    return None


def scene_fields(root: ET.Element) -> dict[str, object]:
    """Extract scene count and scene names from CZI metadata."""
    scenes = root.findall("Metadata/Information/Image/Dimensions/S/Scenes/Scene")
    names = [s.attrib.get("Name", "") for s in scenes]
    return {
        "scene_count": len(scenes),
        "scene_names": ";".join([x for x in names if x]),
    }


def tile_fields(root: ET.Element) -> dict[str, object]:
    """Extract tile region metadata if present in the CZI metadata tree."""
    tile = root.find(".//TileRegion")
    if tile is None:
        return {
            "tile_center_position": None,
            "tile_rows": None,
            "tile_cols": None,
        }
    return {
        "tile_center_position": tile.findtext("CenterPosition"),
        "tile_rows": tile.findtext("Rows"),
        "tile_cols": tile.findtext("Columns"),
    }


def sha256_file(path: Path, chunk_size: int = 1_048_576) -> str:
    """Compute SHA-256 checksum for provenance tracking."""
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def parse_czi_metadata(path: Path, include_sha256: bool = False) -> dict[str, object]:
    """Parse one CZI into a flat metadata dictionary used in the manifest."""
    root = read_embedded_xml(path)
    st = path.stat()

    rec: dict[str, object] = {
        "file_name": path.name,
        "file_path": str(path),
        "size_bytes": st.st_size,
        "created": st.st_ctime,
        "modified": st.st_mtime,
        "acquisition_utc": xtext(
            root, "Metadata/Information/Image/AcquisitionDateAndTime"
        ),
        "pixel_type": xtext(root, "Metadata/Information/Image/PixelType"),
        "size_x": xtext(root, "Metadata/Information/Image/SizeX"),
        "size_y": xtext(root, "Metadata/Information/Image/SizeY"),
        "size_z": xtext(root, "Metadata/Information/Image/SizeZ"),
        "size_c": xtext(root, "Metadata/Information/Image/SizeC"),
        "size_s": xtext(root, "Metadata/Information/Image/SizeS"),
        "size_m": xtext(root, "Metadata/Information/Image/SizeM"),
        "objective_name": xtext(root, ".//ObjectiveName"),
        "objective_mag": xtext(
            root,
            "Metadata/Information/Instrument/Objectives/Objective/NominalMagnification",
        ),
        "objective_na": xtext(
            root, "Metadata/Information/Instrument/Objectives/Objective/LensNA"
        ),
        "camera_name": xtext(root, ".//CameraName"),
        "user_name": xtext(root, ".//UserName"),
        "scale_x_um": scale_axis_um(root, "X"),
        "scale_y_um": scale_axis_um(root, "Y"),
        "scale_z_um": scale_axis_um(root, "Z"),
        "compression": xtext(root, ".//OriginalCompressionMethod"),
    }

    rec.update(channel_fields(root))
    rec.update(scene_fields(root))
    rec.update(tile_fields(root))

    if include_sha256:
        rec["sha256"] = sha256_file(path)

    return rec


def write_manifest_csv(rows: list[dict], out_csv: Path) -> None:
    """Write manifest rows to CSV with stable column ordering."""
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    if not rows:
        out_csv.write_text("", encoding="utf-8")
        return
    fieldnames = list(rows[0].keys())
    with out_csv.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)


def write_manifest_json(rows: list[dict], out_json: Path) -> None:
    """Write manifest rows to pretty-printed JSON."""
    out_json.parent.mkdir(parents=True, exist_ok=True)
    out_json.write_text(json.dumps(rows, indent=2), encoding="utf-8")

In [ ]:
# Parse all CZI files and build manifest table in memory.
if not INPUT_DIR.exists():
    raise RuntimeError(f"Input directory not found: {INPUT_DIR}")

czi_files = sorted(INPUT_DIR.glob("*.czi"))
if not czi_files:
    raise RuntimeError(f"No .czi files found in {INPUT_DIR}")

# Parse each CZI independently so metadata issues are easy to isolate per file.
rows = []
for f in czi_files:
    rec = parse_czi_metadata(f, include_sha256=INCLUDE_SHA256)
    rows.append(rec)
    print("[OK]", f.name)

manifest_df = pd.DataFrame(rows)
manifest_df

[OK] well3-36locations.czi
[OK] well3-568-pSMAD.czi
[OK] well3-647-15BMPbeads.czi
[OK] well3-BF.czi
[OK] well3-DAPI.czi


                  file_name  \
0     well3-36locations.czi   
1       well3-568-pSMAD.czi   
2  well3-647-15BMPbeads.czi   
3              well3-BF.czi   
4            well3-DAPI.czi   

                                           file_path  size_bytes  \
0  <analysis-root>/data_...  1208847456   
1  <analysis-root>/data_...  1574019808   
2  <analysis-root>/data_...  1574013248   
3  <analysis-root>/data_...  1574023360   
4  <analysis-root>/data_...  1574017984   

        created      modified               acquisition_utc pixel_type size_x  \
0  1.771619e+09  1.768834e+09  2026-01-19T13:42:11.1131187Z     Gray16  23372   
1  1.771437e+09  1.768828e+09  2026-01-19T12:52:47.5664545Z     Gray16  13005   
2  1.771437e+09  1.768829e+09  2026-01-19T13:12:05.2732393Z     Gray16  13005   
3  1.771437e+09  1.768827e+09  2026-01-19T12:44:41.9956532Z     Gray16  13005   
4  1.771437e+09  1.768828e+09  2026-01-19T13:03:54.8993067Z     Gray16  13005   

  size_y size_z size_c size_s size_m      

In [ ]:
# Persist manifest outputs for downstream notebooks.
# Persist manifest now so downstream notebooks can consume a stable snapshot.
if WRITE_OUTPUTS:
    write_manifest_csv(rows, OUT_CSV)
    write_manifest_json(rows, OUT_JSON)
    print(f"Wrote {len(rows)} records")
    print("CSV:", OUT_CSV)
    print("JSON:", OUT_JSON)
else:
    print("WRITE_OUTPUTS=False -> not writing CSV/JSON")

Wrote 5 records
CSV: <analysis-root>/pSMAD/results/manifests/czi_manifest.csv
JSON: <analysis-root>/pSMAD/results/manifests/czi_manifest.json


In [ ]:
# QC view focused on dimensions/channels/scenes.
qc_cols = [
    "file_name",
    "size_x",
    "size_y",
    "size_z",
    "size_c",
    "size_s",
    "size_m",
    "scene_count",
    "channel_excitation_nm",
    "channel_emission_nm",
    "scale_x_um",
    "scale_y_um",
    "scale_z_um",
]
manifest_df[qc_cols]

                  file_name size_x size_y size_z size_c size_s size_m  \
0     well3-36locations.czi  23372  23338   None      4     36      1   
1       well3-568-pSMAD.czi  13005  13005      3      1      1    196   
2  well3-647-15BMPbeads.czi  13005  13005      3      1      1    196   
3              well3-BF.czi  13005  13005      3      1      1    196   
4            well3-DAPI.czi  13005  13005      3      1      1    196   

   scene_count channel_excitation_nm channel_emission_nm  scale_x_um  \
0           36           353;577;653         465;603;668        0.65   
1            1                   577                 603        1.30   
2            1                   653                 668        1.30   
3            1                                                  1.30   
4            1                   353                 465        1.30   

   scale_y_um  scale_z_um  
0        0.65         NaN  
1        1.30        50.0  
2        1.30        50.0  
3        1.30   

In [ ]:
# Compact per-file summary for quick sanity checks.
summary_rows = []
for _, r in manifest_df.iterrows():
    summary_rows.append(
        {
            "file_name": r["file_name"],
            "is_36locations": "36locations" in str(r["file_name"]),
            "scene_count": r.get("scene_count", None),
            "size_xy": f"{r.get('size_x', '?')} x {r.get('size_y', '?')}",
            "size_z": r.get("size_z", None),
            "size_c": r.get("size_c", None),
            "scale_x_um": r.get("scale_x_um", None),
            "channels": r.get("channel_ids", ""),
        }
    )

pd.DataFrame(summary_rows).sort_values(
    ["is_36locations", "file_name"], ascending=[False, True]
)

                  file_name  is_36locations  scene_count        size_xy  \
0     well3-36locations.czi            True           36  23372 x 23338   
1       well3-568-pSMAD.czi           False            1  13005 x 13005   
2  well3-647-15BMPbeads.czi           False            1  13005 x 13005   
3              well3-BF.czi           False            1  13005 x 13005   
4            well3-DAPI.czi           False            1  13005 x 13005   

  size_z size_c  scale_x_um                                 channels  
0   None      4        0.65  Channel:0;Channel:1;Channel:2;Channel:3  
1      3      1        1.30                                Channel:0  
2      3      1        1.30                                Channel:0  
3      3      1        1.30                                Channel:0  
4      3      1        1.30                                Channel:0  